<p align="center">
  <span style="color:Navy; font-size:200%; font-weight:bold; vertical-align:middle;">
        Estadística y Probabilidad    
  </span>
</p>
<p align="center" style="line-height:1.2;">
  <span style="color:RoyalBlue; font-size:160%;">Tema 4: Distribuciones muestrales y pruebas de hipótesis </span><br/>
  <span style="color:DodgerBlue; font-size:140%;"> Distribuciones Muestrales y Teorema del Límite Central</span><br/>
  <span style="font-size:100%;color:forestgreen"> Escuela Nacional de Ciencias de la Tierra  |  Semestre 2027-I</span>
</p>

---

# **<font color="Navy">  Laboratorio 4: Distribuciones Muestrales y Teorema del Límite Central </font>**


In [55]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import norm

---
# Parte 1 · <font color = "DarkBlue"> Distribuciones muestrales y Teorema del Límite Central </font>

## ¿Qué es una distribución muestral?

Cuando tomamos una muestra y calculamos un **estadístico** (la media $\bar{x}$, la desviación estándar $s$, una proporción…), ese número depende de *qué* individuos cayeron en la muestra. Si repitiéramos el muestreo, obtendríamos otro valor. Por eso el estadístico **es una variable aleatoria**, y su distribución se llama **distribución muestral**.

## El Teorema del Límite Central (TLC)

Sea una población con media $\mu$ y desviación estándar $\sigma$ **finitas** (de *cualquier* forma). Si tomamos muestras aleatorias de tamaño $n$, entonces, para $n$ suficientemente grande:

$$\bar{X} \;\approx\; N\!\left(\mu,\ \frac{\sigma}{\sqrt{n}}\right) \qquad\Longleftrightarrow\qquad Z = \frac{\bar{X} - \mu}{\sigma/\sqrt{n}} \;\approx\; N(0, 1)$$

Lo sorprendente es la frase "de cualquier forma": aunque los datos individuales sean muy asimétricos o bimodales, **las medias** tienden a una campana normal. La regla práctica "$n \geq 30$" es solo una guía: cuanto más asimétrica la población, más grande debe ser $n$.

## Ejemplo 1.1 — Demostración del TLC con una población muy asimétrica

Sea $X$ el **tiempo (días) entre dos eventos de lluvia intensa** en una región: una variable tipicamente **exponencial**, con media $\mu = 20$ días (y, en la exponencial, $\sigma = \mu = 20$). Es muy asimétrica: muchos tiempos cortos y unos pocos larguísimos. Para cada tamaño $n$ vamos a: tomar 5,000 muestras, calcular la media de cada una, y graficar cómo se distribuyen esas 5,000 medias.

In [ ]:
np.random.seed(2026)

mu_pob = 20        # media poblacional (días)
sigma_pob = 20     # en una exponencial, sigma = media
n_rep = 5000       # número de muestras por cada tamaño n
tamanos = [1, 2, 5, 30, 100]

In [ ]:
medias_por_n = {}
for n in tamanos:
    muestras = np.random.exponential(scale=mu_pob, size=(n_rep, n))
    medias_por_n[n] = muestras.mean(axis=1)


In [ ]:
fig, ejes = plt.subplots(1, 5, figsize=(20, 3.8))
for ax, n in zip(ejes, tamanos):
    m = medias_por_n[n]
    ax.hist(m, bins=35, density=True, color="lightsteelblue", edgecolor="white")
    x_g = np.linspace(max(m.min() - 2, -20), m.max() + 2, 300)
    ax.plot(x_g, norm.pdf(x_g, mu_pob, sigma_pob / np.sqrt(n)), color="crimson", label="Normal (TLC)")
    ax.set_title(f"n = {n}")
    ax.set_xlabel("Media muestral (días)")
ejes[0].set_ylabel("Densidad")
ejes[0].legend()
plt.suptitle("Distribución de 5,000 medias muestrales según el tamaño de la muestra", y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
resumen = pd.DataFrame({
    "n": tamanos,
    "media_de_las_medias": [medias_por_n[n].mean() for n in tamanos],
    "desv_est_de_las_medias": [medias_por_n[n].std() for n in tamanos],
    "teoria_sigma/raiz(n)": [sigma_pob / np.sqrt(n) for n in tamanos],
}).round(2)
resumen

Observa tres cosas:

1. Con $n = 1$ la "distribución de medias" es la población misma: muy asimétrica, y la normal no se parece en nada.
2. Conforme crece $n$, el histograma se vuelve **simétrico y acampanado**, y coincide con la normal que predice el TLC.
3. El centro no cambia ($\approx 20$), pero la **dispersión se reduce**: sigue la fórmula $\sigma/\sqrt{n}$ (tercera columna).

## Ejemplo1.2 — ¿Qué tan rápido converge? Cuantificándolo

Para medir qué tan "normal" es la distribución de las medias, estandarizamos cada media ($Z = (\bar{x}-\mu)/(\sigma/\sqrt{n})$) y calculamos dos indicadores: la **asimetría** (0 si es simétrica) y la **distancia $D$** de Kolmogorov-Smirnov a una normal estándar.

In [ ]:
np.random.seed(7)
tamanos_fino = [1, 2, 5, 10, 20, 30, 50, 100, 200]
asimetria, distancia_D = [], []

for n in tamanos_fino:
    m = np.random.exponential(scale=mu_pob, size=(5000, n)).mean(axis=1)
    z = (m - mu_pob) / (sigma_pob / np.sqrt(n))
    asimetria.append(stats.skew(m))
    distancia_D.append(stats.kstest(z, "norm").statistic)

fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
ejes[0].plot(tamanos_fino, asimetria, "o-", color="darkorange")
ejes[0].axhline(0, color="gray", linestyle="--")
ejes[0].set_xscale("log")
ejes[0].set_xlabel("Tamaño de la muestra n")
ejes[0].set_ylabel("Asimetría de las medias")
ejes[0].set_title("La asimetría desaparece al crecer n")

ejes[1].plot(tamanos_fino, distancia_D, "o-", color="navy")
ejes[1].set_xscale("log")
ejes[1].set_xlabel("Tamaño de la muestra n")
ejes[1].set_ylabel("Distancia D a la normal estándar")
ejes[1].set_title("La distribución se acerca a la normal")
plt.tight_layout()
plt.show()

print(pd.DataFrame({"n": tamanos_fino, "asimetria": np.round(asimetria, 2), "D": np.round(distancia_D, 3)}))

In [ ]:
# Gráficas Q-Q: si los puntos siguen la recta, la distribución es normal
fig, ejes = plt.subplots(1, 3, figsize=(14, 4))
for ax, n in zip(ejes, [2, 5, 30]):
    z = (medias_por_n[n] - mu_pob) / (sigma_pob / np.sqrt(n))
    stats.probplot(z, dist="norm", plot=ax)
    ax.set_title(f"Q-Q de las medias (n = {n})")
plt.tight_layout()
plt.show()

Para esta población, la asimetría de la media baja como $2/\sqrt{n}$: es decir, necesitamos **más** muestra cuanto más asimétrica sea la población. Con $n=30$ ya es bastante pequeña, pero no desaparece del todo.

### Ejercicio para resolver 1.1 — TLC con datos reales (población bimodal)

Las **profundidades focales de los 1000 sismos de Fiji** (km) forman una distribución *bimodal*: un grupo de sismos someros y otro de sismos profundos. Trataremos estos 1000 valores como una **población finita**.

*Fuente: dataset `quakes` (paquete `datasets` de R, servido por Rdatasets).*

```python
url_quakes = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/datasets/quakes.csv"
profundidad = pd.read_csv(url_quakes)["depth"].values
```

1. Calcula la media $\mu$ y la desviación estándar $\sigma$ de la población (usa `np.std` con `ddof=0`, porque es la población completa). Grafica su histograma.
2. Para $n = 2, 10, 50$, toma 5,000 muestras **con reemplazo** (`np.random.choice(profundidad, size=(5000, n))`), calcula la media de cada una y grafica el histograma de las medias junto con la normal $N(\mu, \sigma/\sqrt{n})$.
3. Imprime una tabla que compare la desviación estándar de las medias muestrales con $\sigma/\sqrt{n}$.
4. En una celda de texto: ¿con qué valor de $n$ ya se ve normal? ¿Es sorprendente, dado que la población original tiene dos "jorobas"?

### Ejercicio para resolver 1.2 — TLC con una variable discreta

Sea $X$ el número de **sismos perceptibles en un día** en una estación, con distribución de Poisson de media $\lambda = 0.3$ (la mayoría de los días: 0 sismos). Es discreta y muy asimétrica; para una Poisson, $\mu = \lambda$ y $\sigma = \sqrt{\lambda}$.

1. Para $n = 5, 30, 200$, toma 5,000 muestras de $X$ (`np.random.poisson(0.3, size=(5000, n))`) y calcula la media de cada una.
2. Grafica los 3 histogramas de las medias con la normal $N(\mu, \sigma/\sqrt{n})$ superpuesta.
3. Para $n = 5$, ¿por qué el histograma se ve "dentado" (con huecos entre barras) si usas los `bins` por defecto? ¿Con qué $n$ ya se parece a la normal?

---
# Parte 2 · <font color = "DarkBlue"> Distribución de medias muestrales, error estándar y grados de libertad </font>

## Media y desviación de la media muestral

Si $X_1, \dots, X_n$ son observaciones independientes de una población con media $\mu$ y desviación $\sigma$, entonces:

$$E[\bar{X}] = \mu \qquad\qquad SE(\bar{X}) = \sigma_{\bar{X}} = \frac{\sigma}{\sqrt{n}}$$

- La media muestral es un estimador **insesgado**: en promedio, acierta.
- $SE$ es el **error estándar**: la desviación estándar *del estimador* (no de los datos). Mide qué tanto varía $\bar{x}$ de una muestra a otra.
- Si la población es **normal**, $\bar{X}$ es exactamente normal para cualquier $n$; si no, lo es aproximadamente (TLC).

> **Desviación estándar ≠ error estándar.** La desviación estándar ($\sigma$ o $s$) describe la dispersión de **los datos**; el error estándar ($\sigma/\sqrt{n}$) describe la dispersión de **la media**, y se hace más pequeño al aumentar $n$.

En la práctica, $\sigma$ casi nunca se conoce: se reemplaza por $s$ y se obtiene el **error estándar estimado**: $\;\widehat{SE} = s/\sqrt{n}$.

### Ejemplo 2.1 — Plomo en el suelo: error estándar y probabilidades de la media

Suponemos que en un sitio hay contaminación con plomo: población normal con $\mu = 38$ y $\sigma = 9$ mg/kg. Tomamos muestras de $n = 25$ puntos.

In [ ]:
mu, sigma, n = 38, 9, 25
se_teorico = sigma / np.sqrt(n)
print(f"Error estándar teórico: σ/√n = {se_teorico:.2f} mg/kg")

In [ ]:
np.random.seed(11)
medias = np.random.normal(mu, sigma, size=(10000, n)).mean(axis=1)
print(f"Desviación estándar de 10,000 medias simuladas: {medias.std():.2f}")

In [ ]:
# P(media muestral > 41): teórica (con SE) vs simulada
p_teo = norm.sf(41, loc=mu, scale=se_teorico)
p_sim = np.mean(medias > 41)
print(f"\nP(x̄ > 41):  teórica = {p_teo:.4f}  |  simulada = {p_sim:.4f}")

In [ ]:
np.random.seed(12)
individuales = np.random.normal(mu, sigma, size=10_000)

plt.figure(figsize=(8, 4))
plt.hist(individuales, bins=50, density=True, alpha=0.5, color="gray", label="Valores individuales (σ = 9)")
plt.hist(medias, bins=50, density=True, alpha=0.7, color="steelblue", label=f"Medias de n = 25 (SE = {se_teorico:.1f})")
plt.axvline(mu, color="black", linestyle="--")
plt.xlabel("Concentración de Pb (mg/kg)")
plt.ylabel("Densidad")
plt.title("Los datos individuales varían mucho más que sus medias")
plt.legend()
plt.show()

In [ ]:
# ¿Cómo cambia el error estándar con n? Cada vez que multiplicamos n por 4, el SE se reduce a la mitad.
np.random.seed(13)
ns = [4, 16, 64, 256, 1024]
se_sim = [np.random.normal(mu, sigma, size=(3000, n)).mean(axis=1).std() for n in ns]
se_teo = [sigma / np.sqrt(n) for n in ns]

print(pd.DataFrame({"n": ns, "SE_teorico": np.round(se_teo, 3), "SE_simulado": np.round(se_sim, 3)}))

In [ ]:
plt.figure(figsize=(6.5, 4.2))
plt.loglog(ns, se_teo, "o-", color="crimson", label="σ/√n (teórico)")
plt.loglog(ns, se_sim, "s--", color="navy", label="Simulado")
plt.xlabel("Tamaño de la muestra n")
plt.ylabel("Error estándar")
plt.title("El error estándar decrece como 1/√n")
plt.legend()
plt.show()

### Ejercicio para resolver 2.1

La porosidad efectiva de un acuífero es normal con $\mu = 22\%$ y $\sigma = 4\%$.

1. Calcula el error estándar de la media para $n = 10, 25, 100, 400$.
2. ¿Qué tamaño de muestra se necesita para que el error estándar sea **a lo sumo 0.2**? (Despeja $n$ de $SE = \sigma/\sqrt{n}$.)
3. Para $n = 25$, calcula $P(\bar{X} > 23)$ con la distribución muestral, y compárala con $P(X > 23)$ para **un solo** punto. Verifica la primera con una simulación de 20,000 muestras.

### Ejemplo 2.2 — Error estándar estimado con datos reales (y un experimento de remuestreo)

Con datos reales no conocemos $\sigma$, así que usamos $s$. Tomemos el **caudal anual del río Nilo** (100 años, 1871-1970) como nuestra única muestra.

Además haremos un experimento numérico llamado **remuestreo (bootstrap)**: como solo tenemos *una* muestra, "fingimos" que es la población y sacamos muchas muestras **con reemplazo** de ella (del mismo tamaño $n$). La dispersión de esas medias remuestreadas nos da una estimación *empírica* del error estándar, sin usar la fórmula.

*Fuente: dataset `Nile` (Rdatasets).*

In [ ]:
url_nilo = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/datasets/Nile.csv"
caudal = pd.read_csv(url_nilo)["value"].values

n_n = len(caudal)
xbar = caudal.mean()
s = caudal.std(ddof=1)         # ddof=1: desviación estándar MUESTRAL
se_est = s / np.sqrt(n_n)

print(f"n = {n_n} años")
print(f"Media muestral x̄ = {xbar:.1f}")
print(f"Desviación estándar muestral s = {s:.1f}")
print(f"Error estándar estimado s/√n = {se_est:.1f}")


In [ ]:
# Remuestreo (bootstrap)
np.random.seed(21)
medias_boot = np.random.choice(caudal, size=(5000, n_n), replace=True).mean(axis=1)
print(f"\nDesv. estándar de 5000 medias remuestreadas: {medias_boot.std():.1f}   (fórmula: {se_est:.1f})")

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(medias_boot, bins=40, density=True, color="lightsteelblue", edgecolor="white", label="Medias remuestreadas")
xg = np.linspace(xbar - 4 * se_est, xbar + 4 * se_est, 300)
plt.plot(xg, norm.pdf(xg, xbar, se_est), color="crimson", label="Normal(x̄, s/√n)")
plt.xlabel("Caudal medio anual")
plt.ylabel("Densidad")
plt.title("Distribución (remuestreada) de la media del caudal del Nilo")
plt.legend()
plt.show()

## Grados de libertad: ¿por qué dividimos entre $n-1$?

Para estimar la varianza poblacional $\sigma^2$ usamos la **varianza muestral**:

$$s^2 = \frac{1}{n-1}\sum_{i=1}^{n}(x_i - \bar{x})^2$$

Pero lo "natural" sería dividir entre $n$. ¿Por qué $n-1$? Porque las desviaciones $x_i - \bar{x}$ se calculan respecto a $\bar{x}$, y **siempre suman cero**. Así, si conoces $n-1$ de ellas, la última queda determinada: solo hay $n-1$ piezas de información **libres**. A ese número se le llama **grados de libertad** (*df*, *degrees of freedom*).

Dividir entre $n$ subestima sistemáticamente la varianza (porque $\bar{x}$ está "pegada" a los datos de la muestra, y los datos están siempre más cerca de su propia media que de la media poblacional). Verifiquémoslo con un experimento.

### Ejemplo 2.3

In [ ]:
# 1) Las desviaciones respecto a la media muestral suman cero
np.random.seed(3)
una_muestra = np.random.normal(38, 9, size=5)
desviaciones = una_muestra - una_muestra.mean()
print("Muestra:", np.round(una_muestra, 2))
print("Desviaciones x_i - x̄:", np.round(desviaciones, 2))
print("Suma de las desviaciones:", round(desviaciones.sum(), 10), "   -> la última queda determinada por las otras 4: df = n - 1 = 4")

In [ ]:
# 2) Experimento: 50,000 muestras de n = 5 de una normal con sigma^2 = 81
np.random.seed(5)
sigma2_verd = 81
n = 5
muestras = np.random.normal(38, 9, size=(50_000, n))

var_div_n = muestras.var(axis=1, ddof=0)      # divide entre n
var_div_n1 = muestras.var(axis=1, ddof=1)     # divide entre n-1

print(f"Varianza poblacional verdadera:        {sigma2_verd}")
print(f"Promedio de s² dividiendo entre n:     {var_div_n.mean():.1f}   (esperado: (n-1)/n · σ² = {(n-1)/n * sigma2_verd:.1f})")
print(f"Promedio de s² dividiendo entre n-1:   {var_div_n1.mean():.1f}   <- ¡insesgado!")


In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(["Divide entre n", "Divide entre n-1"], [var_div_n.mean(), var_div_n1.mean()], color=["tomato", "seagreen"])
plt.axhline(sigma2_verd, color="black", linestyle="--", label="σ² verdadera = 81")
plt.ylabel("Promedio de la varianza muestral")
plt.title("Con n = 5, dividir entre n subestima σ²")
plt.legend()
plt.show()

### Ejercicio para resolver 2.3

Repite el experimento para $n = 2, 3, 5, 10, 30, 100$ (usa 20,000 muestras de una $N(38, 9)$ para cada $n$).

1. Para cada $n$, calcula el promedio de la varianza muestral dividiendo entre $n$ y entre $n-1$.
2. Calcula el cociente (promedio con `ddof=0`) / $\sigma^2$ y compáralo con $(n-1)/n$.
3. Grafica ambos cocientes contra $n$ (escala logarítmica en $x$). ¿Para qué tamaños de muestra importa más la corrección de $n-1$?

---
# Parte 3 · <font color = "DarkBlue"> Otras distribuciones muestrales: t de Student, Ji cuadrada y F </font>

Si la población es **normal**, hay tres distribuciones exactas que describen los estadísticos más importantes. Todas se construyen a partir de la normal:

| Distribución | Construcción | Estadístico muestral típico |
|---|---|---|
| **Ji cuadrada** $\chi^2_k$ | Suma de $k$ normales estándar al cuadrado: $\sum_{i=1}^k Z_i^2$ | $\dfrac{(n-1)S^2}{\sigma^2} \sim \chi^2_{n-1}$ |
| **t de Student** $t_k$ | $\dfrac{Z}{\sqrt{\chi^2_k / k}}$ | $\dfrac{\bar{X}-\mu}{S/\sqrt{n}} \sim t_{n-1}$ |
| **F** $F(a, b)$ | $\dfrac{\chi^2_a / a}{\chi^2_b / b}$ | $\dfrac{S_1^2/\sigma_1^2}{S_2^2/\sigma_2^2} \sim F(n_1-1,\ n_2-1)$ |

Los grados de libertad de cada una son los $n-1$ de la sección anterior.

## 3.1 La distribución t de Student

Si conociéramos $\sigma$, el estadístico $Z = (\bar{X}-\mu)/(\sigma/\sqrt{n})$ sería normal estándar. Pero al reemplazar $\sigma$ por $s$ (que también varía de muestra a muestra) agregamos incertidumbre: la distribución resultante es una **t con $n-1$ grados de libertad**, con **colas más pesadas** que la normal. Cuanto mayor $n$, más se parece a la normal.

### Ejemplo 3.1 — Simulando el estadístico $T$ con muestras pequeñas

In [ ]:
np.random.seed(21)
mu, sigma, n = 38, 9, 5
muestras = np.random.normal(mu, sigma, size=(50_000, n))

xbar = muestras.mean(axis=1)
s = muestras.std(axis=1, ddof=1)

Z = (xbar - mu) / (sigma / np.sqrt(n))     # con sigma conocida
T = (xbar - mu) / (s / np.sqrt(n))         # con s estimada

In [ ]:
x_g = np.linspace(-8, 8, 500)
plt.figure(figsize=(9, 4.5))
plt.hist(T, bins=120, range=(-8, 8), density=True, color="lightsteelblue", edgecolor="white", label="T simulado (n = 5)")
plt.plot(x_g, stats.t.pdf(x_g, df=n - 1), color="crimson", linewidth=2, label="t de Student (4 g.l.)")
plt.plot(x_g, norm.pdf(x_g), color="black", linestyle="--", label="Normal estándar")
plt.xlim(-8, 8)
plt.xlabel("Valor del estadístico")
plt.ylabel("Densidad")
plt.title("El estadístico T con n = 5 sigue una t, no una normal")
plt.legend()
plt.show()

In [ ]:
print("Probabilidad en las colas, P(|estadístico| > 2):")
print(f"  T simulado (s estimada):      {np.mean(np.abs(T) > 2):.4f}")
print(f"  t de Student teórica (4 g.l.): {2 * stats.t.sf(2, df=n - 1):.4f}")
print(f"  Z simulado (σ conocida):      {np.mean(np.abs(Z) > 2):.4f}")
print(f"  Normal estándar teórica:      {2 * norm.sf(2):.4f}")

Al estimar $\sigma$ con solo 5 datos, la probabilidad de valores extremos ($|T| > 2$) es **más del doble** de lo que predice la normal (≈ 0.116 vs. 0.046). Si usáramos la normal con $n$ pequeño, subestimaríamos la incertidumbre. Veamos cómo se acerca la t a la normal conforme crecen los grados de libertad:

In [ ]:
x_g = np.linspace(-5, 5, 400)
plt.figure(figsize=(8, 4.2))
for gl, c in zip([1, 4, 10, 30], ["tomato", "darkorange", "seagreen", "navy"]):
    plt.plot(x_g, stats.t.pdf(x_g, gl), color=c, label=f"t con {gl} g.l.")
plt.plot(x_g, norm.pdf(x_g), "k--", label="Normal estándar")
plt.title("La t de Student se aproxima a la normal al aumentar los grados de libertad")
plt.legend()
plt.show()

In [ ]:
# Valores críticos: el valor c tal que P(-c < T < c) = 0.95
tabla_crit = pd.DataFrame({
    "g.l.": [2, 4, 9, 29, 99, 999],
    "t crítico (95%)": [stats.t.ppf(0.975, gl) for gl in [2, 4, 9, 29, 99, 999]],
}).round(3)
tabla_crit["z crítico (95%)"] = round(norm.ppf(0.975), 3)
tabla_crit

### Ejercicio para resolver 3.1

1. Repite la simulación del estadístico $T$ para $n = 3$ (2 g.l.) y $n = 30$ (29 g.l.), con una población $N(38, 9)$ y 50,000 muestras cada una.
2. Para cada $n$, compara la probabilidad $P(|T| > 2.5)$ **simulada**, la **teórica** de la t (`2 * stats.t.sf(2.5, df)`) y la de la **normal** estándar.
3. Calcula con un bucle `for` (o `np.where`) el menor número de grados de libertad para el cual el valor crítico $t_{0.975}$ queda **a menos de 1%** del valor crítico normal (1.96). ¿Qué te dice esto sobre la regla "$n \geq 30$"?

## 3.2 La distribución Ji cuadrada ($\chi^2$)

Si $Z_1, \dots, Z_k$ son normales estándar independientes, entonces $\sum Z_i^2 \sim \chi^2_k$, con $E = k$ y $Var = 2k$. Sus valores son siempre positivos y es asimétrica a la derecha (más asimétrica con pocos g.l.).

### Ejemplo 3.2 — Construyendo la $\chi^2$ sumando cuadrados

In [ ]:
np.random.seed(31)
ks = [1, 2, 5, 10]

fig, ejes = plt.subplots(1, 4, figsize=(18, 3.8))
for ax, k in zip(ejes, ks):
    suma_cuadrados = (np.random.normal(size=(50_000, k)) ** 2).sum(axis=1)
    ax.hist(suma_cuadrados, bins=80, density=True, color="lightsteelblue", edgecolor="white")
    x_g = np.linspace(0.01, suma_cuadrados.max() * 0.8, 400)
    ax.plot(x_g, stats.chi2.pdf(x_g, df=k), color="crimson")
    ax.set_xlim(0, 4 * k + 8)
    ax.set_ylim(0, min(stats.chi2.pdf(x_g, df=k).max() * 1.3, 0.6))
    ax.set_title(f"k = {k} normales al cuadrado")
    print(f"k = {k:2d}: media simulada = {suma_cuadrados.mean():5.2f} (teórica {k})  |  varianza simulada = {suma_cuadrados.var():5.2f} (teórica {2 * k})")
plt.tight_layout()
plt.show()

### Ejemplo 3.3 — La varianza muestral sigue una $\chi^2$

Para muestras de una población **normal**: $\;\dfrac{(n-1)S^2}{\sigma^2} \sim \chi^2_{n-1}$. Verifiquémoslo con $n = 10$ muestras de la $N(38, 9)$ y calculemos la probabilidad de que $S^2$ supere en 50% a la varianza verdadera.

In [ ]:
np.random.seed(41)
mu, sigma, n = 38, 9, 10
muestras = np.random.normal(mu, sigma, size=(50_000, n))

s2 = muestras.var(axis=1, ddof=1)
estadistico = (n - 1) * s2 / sigma ** 2


In [ ]:
x_g = np.linspace(0.01, 35, 400)
plt.figure(figsize=(8, 4.2))
plt.hist(estadistico, bins=80, density=True, color="lightsteelblue", edgecolor="white", label="(n-1)S²/σ² simulado")
plt.plot(x_g, stats.chi2.pdf(x_g, df=n - 1), color="crimson", linewidth=2, label="χ² con 9 g.l.")
plt.xlabel("(n-1)S²/σ²")
plt.ylabel("Densidad")
plt.title("La varianza muestral escalada sigue una Ji cuadrada")
plt.legend()
plt.show()

In [ ]:

print(f"Media simulada = {estadistico.mean():.2f} (teórica {n - 1})  |  Varianza simulada = {estadistico.var():.2f} (teórica {2 * (n - 1)})")

# P(S² > 1.5 σ²)  <=>  P(chi² > 9 · 1.5)
umbral = (n - 1) * 1.5
print(f"\nP(S² > 1.5σ²):  teórica = {stats.chi2.sf(umbral, df=n - 1):.4f}  |  simulada = {np.mean(s2 > 1.5 * sigma**2):.4f}")

Es decir: con solo 10 datos, ¡hay ~14% de probabilidad de que la varianza muestral exceda en 50% a la verdadera! La $\chi^2$ nos permite **cuantificar** cuánto puede fluctuar $S^2$.

### Ejercicio para resolver 3.2

La porosidad de un acuífero es normal con $\mu = 22$ y $\sigma = 4$. Se toma una muestra de $n = 20$ puntos.

1. Calcula la probabilidad (teórica, con `stats.chi2`) de que $S^2 > 2\sigma^2$.
2. Encuentra el intervalo que contiene al 95% central de los valores posibles de $S^2/\sigma^2$ (usa `stats.chi2.ppf(0.025, df)` y `ppf(0.975, df)`, divididos entre $n-1$).
3. Verifica ambos resultados con una simulación de 50,000 muestras.

## 3.3 La distribución F

Si comparamos la variabilidad de **dos** muestras normales independientes, de tamaños $n_1$ y $n_2$, **con la misma varianza poblacional**, el cociente de sus varianzas muestrales sigue una **F** con $(n_1-1,\ n_2-1)$ grados de libertad:

$$F = \frac{S_1^2}{S_2^2} \sim F(n_1 - 1,\ n_2 - 1)$$

Es asimétrica a la derecha, con valores positivos; el valor "típico" está cerca de 1 (varianzas parecidas). Sirve para preguntarnos: *¿qué tan raro sería un cociente de varianzas tan grande como el que observamos, si ambas poblaciones fueran igual de variables?*

### Ejemplo 3.4 — Simulando el cociente de varianzas

In [ ]:
np.random.seed(51)
n1, n2 = 8, 12
reps = 50_000

m1 = np.random.normal(38, 9, size=(reps, n1))
m2 = np.random.normal(38, 9, size=(reps, n2))      # misma varianza poblacional
F_sim = m1.var(axis=1, ddof=1) / m2.var(axis=1, ddof=1)

x_g = np.linspace(0.01, 8, 400)
plt.figure(figsize=(8, 4.2))
plt.hist(F_sim, bins=120, range=(0, 8), density=True, color="lightsteelblue", edgecolor="white", label="S1²/S2² simulado")
plt.plot(x_g, stats.f.pdf(x_g, n1 - 1, n2 - 1), color="crimson", linewidth=2, label="F(7, 11)")
plt.xlabel("Cociente de varianzas muestrales")
plt.ylabel("Densidad")
plt.title("Cociente de varianzas de dos muestras con la misma σ²")
plt.legend()
plt.show()

print(f"P(F > 3):  teórica = {stats.f.sf(3, n1 - 1, n2 - 1):.4f}  |  simulada = {np.mean(F_sim > 3):.4f}")
print(f"Valor crítico: el 95% de los cocientes queda por debajo de {stats.f.ppf(0.95, n1 - 1, n2 - 1):.2f}")

### Ejemplo — Relaciones entre las distribuciones

Todo está conectado: la $t$, la $\chi^2$ y la $F$ salen de la normal. Dos relaciones que se pueden comprobar numéricamente:

- $t_k^2 \sim F(1, k)$ : el cuadrado de una $t$ es una $F$ con $(1, k)$ g.l.
- Si $F \sim F(a, b)$, entonces $1/F \sim F(b, a)$.

In [ ]:
gl = 9
print("Relación 1: (t crítico)² = F crítico con (1, gl) g.l.")
print(f"  t_0.975({gl})² = {stats.t.ppf(0.975, gl) ** 2:.4f}   |   F_0.95(1, {gl}) = {stats.f.ppf(0.95, 1, gl):.4f}")

print("\nRelación 2: el percentil 95 de F(14, 9) es el inverso del percentil 5 de F(9, 14)")
print(f"  F_0.95(14, 9) = {stats.f.ppf(0.95, 14, 9):.4f}   |   1 / F_0.05(9, 14) = {1 / stats.f.ppf(0.05, 9, 14):.4f}")

### Ejemplo 3.5 — Datos reales: ¿cambió la variabilidad del Nilo?

En 1898 se inició la construcción de la presa de Asuán, y el caudal medio del Nilo cambió notablemente. Comparemos el caudal **antes de 1898** ($n_1 = 27$ años) y **desde 1898** ($n_2 = 73$ años). La media cambió mucho — pero ¿cambió también la **variabilidad**?

In [ ]:
nilo = pd.read_csv(url_nilo)[["time", "value"]]
antes = nilo.loc[nilo["time"] < 1898, "value"].values
despues = nilo.loc[nilo["time"] >= 1898, "value"].values

print(f"Antes de 1898:  n = {len(antes):2d} | media = {antes.mean():7.1f} | varianza = {antes.var(ddof=1):8.0f}")
print(f"Desde 1898:     n = {len(despues):2d} | media = {despues.mean():7.1f} | varianza = {despues.var(ddof=1):8.0f}")

In [ ]:
F_obs = antes.var(ddof=1) / despues.var(ddof=1)
gl1, gl2 = len(antes) - 1, len(despues) - 1
print(f"\nCociente de varianzas observado: F = {F_obs:.3f}")
print(f"P(F > F_obs) si las varianzas fueran iguales = {stats.f.sf(F_obs, gl1, gl2):.3f}")
print(f"95% central de la F({gl1}, {gl2}): [{stats.f.ppf(0.025, gl1, gl2):.2f}, {stats.f.ppf(0.975, gl1, gl2):.2f}]")

El cociente observado (≈ 1.17) cae cómodamente dentro del 95% central de la $F(26, 72)$ y su probabilidad de cola (~0.30) no es pequeña: **un cociente así es perfectamente normal si ambas varianzas fueran iguales**. Es decir, la presa cambió el **nivel medio** del caudal pero no hay evidencia de un cambio en su **variabilidad**. 

> Los resultados exactos de $\chi^2$ y $F$ suponen poblaciones **normales** y son más sensibles a esa suposición que la $t$ para la media. Lo veremos en el reto final.

### Ejercicio para resolver 3.3

Se comparan las varianzas de la porosidad en dos pozos: $n_1 = 15$ y $n_2 = 10$ puntos, ambos de una población $N(22, 4)$ (misma varianza).

1. Calcula $P(S_1^2/S_2^2 > 3)$ con `stats.f.sf` y verifícalo con una simulación de 50,000 pares de muestras.
2. Encuentra el valor crítico $F_{0.95}(14, 9)$.
3. Comprueba numéricamente que $F_{0.95}(14, 9) = 1/F_{0.05}(9, 14)$.